In [1]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

#best architecture
architecture = [64, 128, 256, 512, 1024]

# best batch size
batch_size = 128

# best number of epochs
epochs = 15

# define LitNetwork with learning rate scheduler support
class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, use_scheduler=False, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.use_scheduler = use_scheduler
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        
        if self.use_scheduler:
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch"
                }
            }
        else:
            return optimizer

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset   = torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset  = torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# Test different learning rates
learning_rates = [1e-3]
results = []

device = "cpu"

# Test learning rates with scheduler
print("\n" + "="*80)
print("PART 2: LEARNING RATES WITH SCHEDULER (gamma=0.1, step_size=10)")
print("="*80)

for ss in [5, 10, 25]:
    for lr in learning_rates:
        print(f"\nTraining with learning rate: {lr} (with scheduler)")
    
        model = LitNetwork(n_classes=targets_cats, input_ch=features_count, 
                      hidden_sizes=architecture, learning_rate=lr, 
                      use_scheduler=True, scheduler_step_size=ss)

        checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
        logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name=f"lr_{lr:.0e}_with_scheduler")

        start_time = time.time()
        trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
        trainer.fit(model, train_loader, val_loader)
        elapsed_time = time.time() - start_time

        test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
        test_acc = test_results[0]['test_acc']
    
        results.append({
            "learning_rate": lr,
            "scheduler": "StepLR(step=10, gamma=0.1)",
            "training_time": elapsed_time,
            "test_accuracy": test_acc
        })
    
        print(f"Test accuracy: {test_acc:.4f}, Time: {elapsed_time:.2f}s")

# Display summary
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
print(f"{'LR':>10} | {'Scheduler':<30} | {'Time (s)':>10} | {'Test Acc':>10}")
print("-"*80)
for result in results:
    print(f"{result['learning_rate']:>10.0e} | {result['scheduler']:<30} | "
          f"{result['training_time']:>10.2f} | {result['test_accuracy']:>10.4f}")

# Find best result
best_result = max(results, key=lambda x: x['test_accuracy'])
print("\n" + "="*80)
print(f"BEST RESULT: LR={best_result['learning_rate']:.0e}, "
      f"Scheduler={best_result['scheduler']}, "
      f"Accuracy={best_result['test_accuracy']:.4f}")
print("="*80)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores



PART 2: LEARNING RATES WITH SCHEDULER (gamma=0.1, step_size=10)

Training with learning rate: 0.001 (with scheduler)



  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | network   | Sequential         | 706 K  | train
1 | loss_func | CrossEntropyLoss   | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 14: 100%|██████████| 3178/3178 [01:24<00:00, 37.43it/s, v_num=1, val_acc=0.942]

...

Restoring states from the checkpoint path at my_logs\lr_1e-03_with_scheduler\version_3\checkpoints\epoch=14-step=47670.ckpt


Loaded model weights from the checkpoint at my_logs\lr_1e-03_with_scheduler\version_3\checkpoints\epoch=14-step=47670.ckpt


Testing DataLoader 0: 100%|██████████| 908/908 [00:07<00:00, 114.86it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9334957003593445     │
└───────────────────────────┴───────────────────────────┘

Test accuracy: 0.9335, Time: 1240.36s

RESULTS SUMMARY
        LR | Scheduler                      |   Time (s) |   Test Acc
--------------------------------------------------------------------------------
     1e-03 | StepLR(step=10, gamma=0.1)     |    1402.66 |     0.9419
     1e-03 | StepLR(step=10, gamma=0.1)     |    1247.88 |     0.9547
     1e-03 | StepLR(step=10, gamma=0.1)     |    1240.36 |     0.9335

BEST RESULT: LR=1e-03, Scheduler=StepLR(step=10, gamma=0.1), Accuracy=0.9547


In [3]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

# configurable variables
architecture = [64, 128, 256, 512, 1024]
batch_size = 128
epochs = 15
lr = 1e-3
ss = 8
device = "cpu"

# define LitNetwork with learning rate scheduler support
class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, use_scheduler=False, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.use_scheduler = use_scheduler
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        
        if self.use_scheduler:
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch"
                }
            }
        else:
            return optimizer

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset =   torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset =  torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader =  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = LitNetwork(n_classes=targets_cats, input_ch=features_count, 
                   hidden_sizes=architecture, learning_rate=lr, 
                   use_scheduler=True, scheduler_step_size=ss)

checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name=f"final")

start_time = time.time()

trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
trainer.fit(model, train_loader, val_loader)

elapsed_time = time.time() - start_time

test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
test_acc = test_results[0]['test_acc']

results = {
    "architecture": architecture,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "scheduler": (print(f"StepLR(step={ss}, gamma=0.1)")),
    "training_time": elapsed_time,
    "test_accuracy": test_acc
}

print(results)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | network   | Sequential         | 706 K  | train
1 | loss_func | CrossEntropyLoss   | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 14: 100%|██████████| 3178/3178 [01:46<00:00, 29.94it/s, v_num=1, val_acc=0.953]

`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 3178/3178 [01:46<00:00, 29.92it/s, v_num=1, val_acc=0.953]


Restoring states from the checkpoint path at my_logs\final\version_1\checkpoints\epoch=14-step=47670.ckpt
Loaded model weights from the checkpoint at my_logs\final\version_1\checkpoints\epoch=14-step=47670.ckpt
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 908/908 [00:11<00:00, 82.31it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.953899621963501     │
└───────────────────────────┴───────────────────────────┘

StepLR(step=8, gamma=0.1)
{'architecture': [64, 128, 256, 512, 1024], 'batch_size': 128, 'epochs': 15, 'learning_rate': 0.001, 'scheduler': None, 'training_time': 1657.2861268520355, 'test_accuracy': 0.953899621963501}


In [ ]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

# configurable variables
architecture = [64, 128, 256, 512, 1024]
batch_size = 128
epochs = 15
lr = 1e-3
ss = 9
device = "cpu"

# define LitNetwork with learning rate scheduler support
class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, use_scheduler=False, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.use_scheduler = use_scheduler
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        
        if self.use_scheduler:
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch"
                }
            }
        else:
            return optimizer

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset =   torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset =  torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader =  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = LitNetwork(n_classes=targets_cats, input_ch=features_count, 
                   hidden_sizes=architecture, learning_rate=lr, 
                   use_scheduler=True, scheduler_step_size=ss)

checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name=f"final")

start_time = time.time()

trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
trainer.fit(model, train_loader, val_loader)

elapsed_time = time.time() - start_time

test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
test_acc = test_results[0]['test_acc']

results = {
    "architecture": architecture,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "scheduler": "StepLR",
    "step": ss,
    "training_time": elapsed_time,
    "test_accuracy": test_acc
}

print(results)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | network   | Sequential         | 706 K  | train
1 | loss_func | CrossEntropyLoss   | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 14: 100%|██████████| 3178/3178 [01:39<00:00, 31.82it/s, v_num=2, val_acc=0.954]

`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 3178/3178 [01:39<00:00, 31.80it/s, v_num=2, val_acc=0.954]

Restoring states from the checkpoint path at my_logs\final\version_2\checkpoints\epoch=14-step=47670.ckpt


Loaded model weights from the checkpoint at my_logs\final\version_2\checkpoints\epoch=14-step=47670.ckpt
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 908/908 [00:09<00:00, 98.32it/s] 


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9530649185180664     │
└───────────────────────────┴───────────────────────────┘

StepLR(step=9, gamma=0.1)
{'architecture': [64, 128, 256, 512, 1024], 'batch_size': 128, 'epochs': 15, 'learning_rate': 0.001, 'scheduler': None, 'training_time': 1486.2012393474579, 'test_accuracy': 0.9530649185180664}


In [6]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

# configurable variables
architecture = [64, 128, 256, 512, 1024]
batch_size = 128
epochs = 15
lr = 1e-3
ss = 10
device = "cpu"

# define LitNetwork with learning rate scheduler support
class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, use_scheduler=False, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.use_scheduler = use_scheduler
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        
        if self.use_scheduler:
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch"
                }
            }
        else:
            return optimizer

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset =   torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset =  torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader =  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = LitNetwork(n_classes=targets_cats, input_ch=features_count, 
                   hidden_sizes=architecture, learning_rate=lr, 
                   use_scheduler=True, scheduler_step_size=ss)

checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name=f"final")

start_time = time.time()

trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
trainer.fit(model, train_loader, val_loader)

elapsed_time = time.time() - start_time

test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
test_acc = test_results[0]['test_acc']

results = {
    "architecture": architecture,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "scheduler": "StepLR",
    "step": ss,
    "training_time": elapsed_time,
    "test_accuracy": test_acc
}

print(results)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | network   | Sequential         | 706 K  | train
1 | loss_func | CrossEntropyLoss   | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 14: 100%|██████████| 3178/3178 [01:26<00:00, 36.74it/s, v_num=4, val_acc=0.954]

`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 3178/3178 [01:26<00:00, 36.72it/s, v_num=4, val_acc=0.954]


Restoring states from the checkpoint path at my_logs\final\version_4\checkpoints\epoch=14-step=47670.ckpt
Loaded model weights from the checkpoint at my_logs\final\version_4\checkpoints\epoch=14-step=47670.ckpt
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 908/908 [00:08<00:00, 107.93it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9545708894729614     │
└───────────────────────────┴───────────────────────────┘

{'architecture': [64, 128, 256, 512, 1024], 'batch_size': 128, 'epochs': 15, 'learning_rate': 0.001, 'scheduler': 'StepLR', 'step': 10, 'training_time': 1307.7562019824982, 'test_accuracy': 0.9545708894729614}


In [7]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

# configurable variables
architecture = [64, 128, 256, 512, 1024]
batch_size = 128
epochs = 15
lr = 1e-3
ss = 12
device = "cpu"

# define LitNetwork with learning rate scheduler support
class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, use_scheduler=False, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.use_scheduler = use_scheduler
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        
        if self.use_scheduler:
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch"
                }
            }
        else:
            return optimizer

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset =   torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset =  torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader =  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = LitNetwork(n_classes=targets_cats, input_ch=features_count, 
                   hidden_sizes=architecture, learning_rate=lr, 
                   use_scheduler=True, scheduler_step_size=ss)

checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name="final")

start_time = time.time()

trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
trainer.fit(model, train_loader, val_loader)

elapsed_time = time.time() - start_time

test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
test_acc = test_results[0]['test_acc']

results = {
    "architecture": architecture,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "scheduler": "StepLR",
    "step": ss,
    "training_time": elapsed_time,
    "test_accuracy": test_acc
}

print(results)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | network   | Sequential         | 706 K  | train
1 | loss_func | CrossEntropyLoss   | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 14: 100%|██████████| 3178/3178 [01:22<00:00, 38.72it/s, v_num=5, val_acc=0.929]

`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 3178/3178 [01:22<00:00, 38.70it/s, v_num=5, val_acc=0.929]

Restoring states from the checkpoint path at my_logs\final\version_5\checkpoints\epoch=14-step=47670.ckpt


Loaded model weights from the checkpoint at my_logs\final\version_5\checkpoints\epoch=14-step=47670.ckpt
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 908/908 [00:07<00:00, 114.59it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.929270327091217     │
└───────────────────────────┴───────────────────────────┘

{'architecture': [64, 128, 256, 512, 1024], 'batch_size': 128, 'epochs': 15, 'learning_rate': 0.001, 'scheduler': 'StepLR', 'step': 15, 'training_time': 1188.3987383842468, 'test_accuracy': 0.929270327091217}


In [8]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

# configurable variables
architecture = [64, 128, 256, 512, 1024]
batch_size = 128
epochs = 15
lr = 1e-3
ss = 15
device = "cpu"

# define LitNetwork with learning rate scheduler support
class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, use_scheduler=False, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.use_scheduler = use_scheduler
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        
        if self.use_scheduler:
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch"
                }
            }
        else:
            return optimizer

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset =   torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset =  torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader =  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = LitNetwork(n_classes=targets_cats, input_ch=features_count, 
                   hidden_sizes=architecture, learning_rate=lr, 
                   use_scheduler=True, scheduler_step_size=ss)

checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name="final")

start_time = time.time()

trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
trainer.fit(model, train_loader, val_loader)

elapsed_time = time.time() - start_time

test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
test_acc = test_results[0]['test_acc']

results = {
    "architecture": architecture,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "scheduler": "StepLR",
    "step": ss,
    "training_time": elapsed_time,
    "test_accuracy": test_acc
}

print(results)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | network   | Sequential         | 706 K  | train
1 | loss_func | CrossEntropyLoss   | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 14: 100%|██████████| 3178/3178 [01:21<00:00, 38.85it/s, v_num=6, val_acc=0.928]

`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 3178/3178 [01:21<00:00, 38.85it/s, v_num=6, val_acc=0.928]


Restoring states from the checkpoint path at my_logs\final\version_6\checkpoints\epoch=12-step=41314.ckpt
Loaded model weights from the checkpoint at my_logs\final\version_6\checkpoints\epoch=12-step=41314.ckpt
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 908/908 [00:07<00:00, 115.02it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9310086369514465     │
└───────────────────────────┴───────────────────────────┘

{'architecture': [64, 128, 256, 512, 1024], 'batch_size': 128, 'epochs': 15, 'learning_rate': 0.001, 'scheduler': 'StepLR', 'step': 15, 'training_time': 1191.2415661811829, 'test_accuracy': 0.9310086369514465}


In [9]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

# configurable variables
architecture = [64, 128, 256, 512, 1024]
batch_size = 128
epochs = 15
lr = 1e-3
ss = 11
device = "cpu"

# define LitNetwork with learning rate scheduler support
class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, use_scheduler=False, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.use_scheduler = use_scheduler
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        
        if self.use_scheduler:
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch"
                }
            }
        else:
            return optimizer

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset =   torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset =  torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader =  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = LitNetwork(n_classes=targets_cats, input_ch=features_count, 
                   hidden_sizes=architecture, learning_rate=lr, 
                   use_scheduler=True, scheduler_step_size=ss)

checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name="final")

start_time = time.time()

trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
trainer.fit(model, train_loader, val_loader)

elapsed_time = time.time() - start_time

test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
test_acc = test_results[0]['test_acc']

results = {
    "architecture": architecture,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "scheduler": "StepLR",
    "step": ss,
    "training_time": elapsed_time,
    "test_accuracy": test_acc
}

print(results)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | network   | Sequential         | 706 K  | train
1 | loss_func | CrossEntropyLoss   | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 14: 100%|██████████| 3178/3178 [01:40<00:00, 31.53it/s, v_num=7, val_acc=0.956]

`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 3178/3178 [01:40<00:00, 31.52it/s, v_num=7, val_acc=0.956]


Restoring states from the checkpoint path at my_logs\final\version_7\checkpoints\epoch=14-step=47670.ckpt
Loaded model weights from the checkpoint at my_logs\final\version_7\checkpoints\epoch=14-step=47670.ckpt
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 908/908 [00:09<00:00, 97.26it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.954923689365387     │
└───────────────────────────┴───────────────────────────┘

{'architecture': [64, 128, 256, 512, 1024], 'batch_size': 128, 'epochs': 15, 'learning_rate': 0.001, 'scheduler': 'StepLR', 'step': 11, 'training_time': 1342.0619864463806, 'test_accuracy': 0.954923689365387}


In [ ]:
import time
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load data
df = pd.read_csv("./datasets/forest_cover.csv")

# data preparation
wilderness_cols = [c for c in df.columns if "Wilderness_Area" in c]
if len(wilderness_cols) > 0:
    df["Wilderness"] = df[wilderness_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=wilderness_cols)

soil_cols = [c for c in df.columns if "Soil_Type" in c]
if len(soil_cols) > 0:
    df["Soil"] = df[soil_cols].values.argmax(axis=1) + 1
    df = df.drop(columns=soil_cols)

continuous_cols = [c for c in df.columns if c not in ["Cover_Type", "Wilderness", "Soil"]]
categorical_cols = ["Wilderness", "Soil"]

X = df.drop(columns=['Cover_Type'])
y = df['Cover_Type'].astype(int) - 1

X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
val_frac = 0.10 / 0.80
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, test_size=val_frac, stratify=y_rem, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train[continuous_cols].values)

def preprocess_split(X_df):
    X_cont = scaler.transform(X_df[continuous_cols].values).astype(np.float32)
    X_cat = X_df[categorical_cols].values.astype(np.float32)
    X_combined = np.hstack([X_cont, X_cat])
    return X_combined

X_train_np = preprocess_split(X_train)
X_val_np   = preprocess_split(X_val)
X_test_np  = preprocess_split(X_test)

y_train_np = y_train.to_numpy().astype(np.int64)
y_val_np   = y_val.to_numpy().astype(np.int64)
y_test_np  = y_test.to_numpy().astype(np.int64)

features_count = X_train_np.shape[1]
targets_cats = 7

# configurable variables
architecture = [64, 128, 256, 512, 1024]
batch_size = 128
epochs = 500
lr = 1e-3
ss = 11
device = "cpu"

# define LitNetwork with learning rate scheduler support
class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=7, input_ch=12, hidden_sizes=[128, 64, 32], 
                 learning_rate=1e-3, use_scheduler=False, scheduler_step_size=10):
        super(LitNetwork, self).__init__()
        self.learning_rate = learning_rate
        self.use_scheduler = use_scheduler
        self.scheduler_step_size = scheduler_step_size
        
        layers = []
        prev_size = input_ch
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, n_classes))
        
        self.network = nn.Sequential(*layers)
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        
        if self.use_scheduler:
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=self.scheduler_step_size, gamma=0.1)
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "interval": "epoch"
                }
            }
        else:
            return optimizer

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs, target)
        self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
    
    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs, target)
        self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_train_np), torch.from_numpy(y_train_np))
val_dataset =   torch.utils.data.TensorDataset(torch.from_numpy(X_val_np),   torch.from_numpy(y_val_np))
test_dataset =  torch.utils.data.TensorDataset(torch.from_numpy(X_test_np),  torch.from_numpy(y_test_np))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader =   DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader =  DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

model = LitNetwork(n_classes=targets_cats, input_ch=features_count, 
                   hidden_sizes=architecture, learning_rate=lr, 
                   use_scheduler=True, scheduler_step_size=ss)

checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
logger = pl_loggers.TensorBoardLogger(save_dir="my_logs", name="final")

start_time = time.time()

trainer = pl.Trainer(max_epochs=epochs, accelerator=device, callbacks=[checkpoint], logger=logger)
trainer.fit(model, train_loader, val_loader)

elapsed_time = time.time() - start_time

test_results = trainer.test(ckpt_path="best", dataloaders=test_loader)
test_acc = test_results[0]['test_acc']

results = {
    "architecture": architecture,
    "batch_size": batch_size,
    "epochs": epochs,
    "learning_rate": lr,
    "scheduler": "StepLR",
    "step": ss,
    "training_time": elapsed_time,
    "test_accuracy": test_acc
}

print(results)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | network   | Sequential         | 706 K  | train
1 | loss_func | CrossEntropyLoss   | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
706 K     Trainable params
0         Non-trainable params
706 K     Total params
2.825     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\Braxt\miniconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 265:  45%|████▌     | 1437/3178 [00:36<00:43, 39.66it/s, v_num=11, val_acc=0.963]